# 15.5 選中多位 expert 怎麼合併？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：送出時分組，收回時依原訂單編號**

同token多份貢獻要相加，不能覆蓋。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dispatch.svg")))

**先想一想：**權重0.7、0.2重新除0.9後合計是多少？

有多位被選中時，用gate權重加總回同一個feature向量。常見top-2會重新正規化選中權重；top-1要特別留意梯度問題。

**把直覺寫成數學：**output=Σ_selected gate_e·expert_e(x)；本教材top-1保留原softmax機率。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
weights = torch.tensor([0.7, 0.2])
normalized = weights / weights.sum()
outputs = torch.tensor([[1.0, 0.0], [0.0, 2.0]])
print(normalized, normalized @ outputs)
assert torch.allclose(normalized.sum(), torch.tensor(1.0))

**如何讀結果：**輸出可以不落在原token附近；是否保留residual由外層block負責。

**只改一件事：**只交換兩位expert的輸出。
